# Camino standoff con 7 gases desde YAML

El mismo caso de `run_simulation()` de FORWARD (`final_forward_attenuation.py`), ahora descrito **entero en un YAML** y corrido con `run_path()`: 7 gases con todos sus isotopólogos, camino de 1 m a 1013 hPa y 288.2 K, 600–2500 cm⁻¹ con paso 0.01 cm⁻¹, promediado en bines de 0.02 µm.

| FORWARD (`run_simulation`) | pylblrtm (`examples/standoff_7gases.yaml`) |
|---|---|
| `s.Pmol` (presión parcial, atm) | `path.gases` con `gas_units: atm` |
| `L_m`, `pres`, `temp_K` | `path.length_m`, `path.pressure_hpa`, `path.temperature_k` |
| `use_all_isotopologues=True` | `path.isotopologues: all` |
| `nu_min`, `nu_max`, `dnu` | `spectral.nu_min_cm1`, `nu_max_cm1`, `step_cm1` |
| `delta_um=0.020` | `output.bin_um: 0.02` |
| `att=True` | `output.attenuation_db_m: true` |
| `save_csv=True`, `outdir='OUT'` | `output.save` |
| `res['lambda_centers']` | `res.wavelength_um` |
| `res['T_prod_samp']` | `res.transmittance_binned` |
| `res['T_each_samp'][k]` | `res.transmittance_per_gas_binned["CO2"]` (por nombre) |

Diferencias con FORWARD: las líneas vienen de HITRAN2024 (no de `PARS/ALL.par`) y las TIPS son TIPS-2025; cada línea aporta hasta 25 cm⁻¹ de su centro (FORWARD usa tiles de 20 cm⁻¹ con 5 cm⁻¹ de guard). Ojo: en FORWARD `T_each_samp[0]` es el **H2O** (primer gas de `default_species()`), no el CO2.

Es un cálculo grande: en cvail, `sbatch scripts/run_notebook.sh notebooks/standoff_7gases.ipynb`. Requiere `pip install -e ".[notebook]"` (y `.[fast]` para el backend rápido).

In [ ]:
from pathlib import Path

# Raíz del repo, abriendo el notebook desde ahí o desde notebooks/
REPO = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "examples").is_dir())
YAML = REPO / "examples" / "standoff_7gases.yaml"
print(YAML)

In [ ]:
import pathlib
print(YAML.read_text(encoding="utf-8"))

In [ ]:
import importlib.util
import time
from pylblrtm import load_config, run_path

cfg = load_config(YAML)
# Los CSV de output.save quedan en <repo>/OUT/
cfg["output"]["save"]["path"] = str(REPO / "OUT")
# Con numba (cvail) el backend Humlicek es mucho más rápido y difiere < 1e-4 de wofz
if importlib.util.find_spec("numba") is not None:
    cfg["calculation"]["voigt"] = "humlicek"

import contextlib, io
t0 = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):   # HAPI imprime el progreso de cada descarga
    res = run_path(cfg)
print(res)
print(f"{time.perf_counter() - t0:.0f} s · voigt={cfg['calculation']['voigt']} · "
      f"gases sin líneas en el rango: {res.skipped_gases or 'ninguno'}")

# Los mismos nombres que en FORWARD
lambda_um = res.wavelength_um
Tprod = res.transmittance_binned
Tco2  = res.transmittance_per_gas_binned["CO2"]
print(f"{lambda_um.size} bines de 0.02 µm entre {lambda_um[0]:.3f} y {lambda_um[-1]:.3f} µm")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SERIES, INK, INK_2, MUTED, GRID, AXIS, SURFACE = (
    "#2a78d6", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})

# Un panel por serie: total arriba y cada gas debajo, mismo eje de longitud de onda
panels = [("Total (7 gases)", Tprod)] + list(res.transmittance_per_gas_binned.items())
fig, axes = plt.subplots(len(panels), 1, figsize=(11, 1.6 * len(panels)), sharex=True)
for ax, (name, T) in zip(axes, panels):
    ax.plot(lambda_um, T, color=SERIES, linewidth=1.2)
    ax.set_ylim(-0.02, 1.02)
    ax.set_yticks([0, 0.5, 1])
    ax.text(0.005, 0.08, f"{name} · T mín {T.min():.4g}", transform=ax.transAxes,
            color=INK, fontsize=10, fontweight="bold")
axes[0].set_title(f"Transmitancia en bines de {cfg['output']['bin_um']} µm · "
                  f"L = {cfg['path']['length_m']} m, {cfg['path']['pressure_hpa']} hPa, "
                  f"{cfg['path']['temperature_k']} K", color=INK, loc="left")
axes[-1].set_xlabel("Longitud de onda [µm]")
fig.supylabel("Transmitancia", color=INK_2)
fig.tight_layout()
plt.show()

In [ ]:
# Atenuación total [dB/m] (att=True de FORWARD); por gas en res.attenuation_per_gas_db_m
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.semilogy(lambda_um, np.clip(res.attenuation_db_m, 1e-6, None), color=SERIES, linewidth=1.2)
ax.set_xlabel("Longitud de onda [µm]")
ax.set_ylabel("Atenuación [dB/m]")
ax.set_title("Atenuación total, 7 gases", color=INK, loc="left")
fig.tight_layout()
plt.show()

print("Archivos guardados por output.save:", sorted(p.name for p in (REPO / "OUT").glob("standoff_7gases*")))